# Week 2 — Aido Rover Reinforcement Learning Environment

This notebook implements a custom Gymnasium environment for a simplified Aido Rover autonomous patrol decision task.

The environment models the rover as a Markov Decision Process (MDP) in which the agent observes a synthetic sensor vector and selects one of four high-level operational actions:

- `PATROL` — continue normal patrol
- `ALERT` — stop and broadcast an alert
- `CHARGE` — return to the charging dock
- `INVESTIGATE` — move toward a detected anomaly

The observation vector represents five synthetic sensor channels:

1. Motor current
2. Battery state of charge (SoC)
3. IMU magnitude
4. Proximity
5. RSSI / communication signal strength

The main design goal is to reward safe and useful patrol behavior while penalizing missed faults, unnecessary alerts, and battery depletion.


In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env


## Environment design

The synthetic state is a five-value sensor vector:

`[motor_current, battery_soc, imu_magnitude, proximity, rssi]`

All values are normalized to the range `[0, 1]` to keep the observation space simple and stable for PPO training.

The environment uses four discrete actions:

- `0 = PATROL`
- `1 = ALERT`
- `2 = CHARGE`
- `3 = INVESTIGATE`

A hidden simulated fault state is used to make the reward depend on whether the rover's action is appropriate for the situation, rather than rewarding an action unconditionally.


In [2]:
class AidoRoverEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    PATROL = 0
    ALERT = 1
    CHARGE = 2
    INVESTIGATE = 3

    def __init__(
        self,
        max_steps=200,
        fault_probability=0.05,
        fault_timeout=8,
    ):
        super().__init__()

        self.max_steps = max_steps
        self.fault_probability = fault_probability
        self.fault_timeout = fault_timeout

        self.observation_space = spaces.Box(
            low=np.zeros(5, dtype=np.float32),
            high=np.ones(5, dtype=np.float32),
            dtype=np.float32,
        )

        self.action_space = spaces.Discrete(4)

        self.step_count = 0
        self.fault_active = False
        self.fault_age = 0

        self.motor_current = 0.0
        self.battery_soc = 1.0
        self.imu_magnitude = 0.0
        self.proximity = 1.0
        self.rssi = 1.0

    def _get_observation(self):
        return np.array(
            [
                self.motor_current,
                self.battery_soc,
                self.imu_magnitude,
                self.proximity,
                self.rssi,
            ],
            dtype=np.float32,
        )

    def _get_info(self):
        return {
            "fault_active": self.fault_active,
            "fault_age": self.fault_age,
            "step_count": self.step_count,
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.step_count = 0
        self.fault_active = False
        self.fault_age = 0

        self.motor_current = float(self.np_random.uniform(0.20, 0.40))
        self.battery_soc = float(self.np_random.uniform(0.85, 1.00))
        self.imu_magnitude = float(self.np_random.uniform(0.05, 0.20))
        self.proximity = float(self.np_random.uniform(0.60, 1.00))
        self.rssi = float(self.np_random.uniform(0.70, 1.00))

        return self._get_observation(), self._get_info()

    def _update_sensors(self, action):
        if action == self.CHARGE:
            self.battery_soc = min(1.0, self.battery_soc + 0.08)
        elif action == self.INVESTIGATE:
            self.battery_soc = max(0.0, self.battery_soc - 0.015)
        elif action == self.ALERT:
            self.battery_soc = max(0.0, self.battery_soc - 0.006)
        else:
            self.battery_soc = max(0.0, self.battery_soc - 0.010)

        if (
            not self.fault_active
            and self.np_random.random() < self.fault_probability
        ):
            self.fault_active = True
            self.fault_age = 0

        if self.fault_active:
            self.fault_age += 1
            target_motor = 0.80
            target_imu = 0.75
            target_proximity = 0.20
            target_rssi = 0.45
        else:
            self.fault_age = 0
            target_motor = 0.30
            target_imu = 0.15
            target_proximity = 0.80
            target_rssi = 0.85

        self.motor_current = float(
            np.clip(
                0.7 * self.motor_current
                + 0.3 * target_motor
                + self.np_random.normal(0.0, 0.03),
                0.0,
                1.0,
            )
        )

        self.imu_magnitude = float(
            np.clip(
                0.7 * self.imu_magnitude
                + 0.3 * target_imu
                + self.np_random.normal(0.0, 0.03),
                0.0,
                1.0,
            )
        )

        self.proximity = float(
            np.clip(
                0.7 * self.proximity
                + 0.3 * target_proximity
                + self.np_random.normal(0.0, 0.04),
                0.0,
                1.0,
            )
        )

        self.rssi = float(
            np.clip(
                0.7 * self.rssi
                + 0.3 * target_rssi
                + self.np_random.normal(0.0, 0.03),
                0.0,
                1.0,
            )
        )

    def _calculate_reward(self, action):
        reward = 0.0

        if action == self.CHARGE:
            if self.battery_soc < 0.25:
                reward += 3.0
            else:
                reward -= 1.0

        if self.fault_active:
            if action == self.ALERT:
                reward += 5.0
                self.fault_active = False
                self.fault_age = 0

            elif action == self.INVESTIGATE:
                reward += 4.0

                if self.np_random.random() < 0.70:
                    self.fault_active = False
                    self.fault_age = 0

            elif action == self.PATROL:
                reward -= 3.0

            elif action == self.CHARGE:
                reward -= 2.0

        else:
            if action == self.PATROL:
                reward += 1.0
            elif action == self.ALERT:
                reward -= 1.5
            elif action == self.INVESTIGATE:
                reward -= 0.75

        if self.battery_soc <= 0.0:
            reward -= 10.0

        return reward

    def step(self, action):
        if not self.action_space.contains(action):
            raise ValueError(f"Invalid action: {action}")

        self.step_count += 1

        self._update_sensors(action)
        reward = self._calculate_reward(action)

        terminated = False
        truncated = False

        if self.battery_soc <= 0.0:
            terminated = True

        if self.fault_active and self.fault_age > self.fault_timeout:
            reward -= 8.0
            terminated = True

        if self.step_count >= self.max_steps:
            truncated = True

        return (
            self._get_observation(),
            reward,
            terminated,
            truncated,
            self._get_info(),
        )

    def render(self):
        operating_state = "FAULT" if self.fault_active else "NORMAL"
        print(
            f"step={self.step_count:03d} | "
            f"state={operating_state:6s} | "
            f"battery={self.battery_soc:.2f} | "
            f"motor={self.motor_current:.2f} | "
            f"imu={self.imu_magnitude:.2f} | "
            f"proximity={self.proximity:.2f} | "
            f"rssi={self.rssi:.2f}"
        )

    def close(self):
        pass


## Gymnasium compatibility check

Before training PPO, validate that the environment follows the Gymnasium API correctly.


In [3]:
env = AidoRoverEnv()

check_env(env)

print("Gymnasium environment check passed.")


Gymnasium environment check passed.


c:\Users\User\Desktop\work\ingen-mlnn-week1-starter\.venv\Lib\site-packages\gymnasium\utils\env_checker.py:440: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


## Manual rollout

Run a short random-policy rollout to verify that observations, rewards, faults, battery changes, and episode termination behave sensibly before using Stable Baselines3 PPO.


In [4]:
env = AidoRoverEnv()

observation, info = env.reset(seed=42)

print("Initial observation:", observation)
print("Initial info:", info)

total_reward = 0.0

for _ in range(20):
    action = env.action_space.sample()

    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    print(
        f"action={action} | "
        f"reward={reward:6.2f} | "
        f"obs={np.round(observation, 2)} | "
        f"fault={info['fault_active']}"
    )

    if terminated or truncated:
        break

print(f"Total rollout reward: {total_reward:.2f}")

env.close()


Initial observation: [0.35479122 0.91583174 0.17878969 0.8789472  0.7282532 ]
Initial info: {'fault_active': False, 'fault_age': 0, 'step_count': 0}
action=2 | reward= -1.00 | obs=[0.34 1.   0.16 0.85 0.74] | fault=False
action=3 | reward= -0.75 | obs=[0.35 0.98 0.16 0.88 0.79] | fault=False
action=1 | reward= -1.50 | obs=[0.35 0.97 0.13 0.89 0.8 ] | fault=False
action=3 | reward= -0.75 | obs=[0.31 0.96 0.17 0.86 0.8 ] | fault=False
action=2 | reward= -1.00 | obs=[0.33 1.   0.18 0.86 0.83] | fault=False
action=0 | reward=  1.00 | obs=[0.31 0.99 0.15 0.81 0.86] | fault=False
action=2 | reward= -1.00 | obs=[0.3  1.   0.13 0.77 0.87] | fault=False
action=3 | reward= -0.75 | obs=[0.32 0.98 0.11 0.79 0.87] | fault=False
action=1 | reward= -1.50 | obs=[0.34 0.98 0.13 0.82 0.87] | fault=False
action=0 | reward=  1.00 | obs=[0.35 0.97 0.09 0.8  0.85] | fault=False
action=3 | reward= -0.75 | obs=[0.32 0.95 0.16 0.77 0.88] | fault=False
action=3 | reward= -0.75 | obs=[0.31 0.94 0.16 0.8  0.89] |

## Design summary

This environment is intentionally a simplified, synthetic representation of an Aido Rover patrol decision task.

The key difference from the earlier Breakout environment is the source of the state and reward:

- Breakout observations come from the game state, while this environment uses structured robot sensor values.
- Breakout provides a game-defined score, while the Aido Rover reward must be designed from operational goals.
- The action space remains discrete so PPO can be applied in a directly comparable way.
- A real robot deployment would add more complex sensor noise, continuous or hybrid controls, real-time constraints, and safety requirements.

The reward design will be revisited after PPO training to determine whether the learned behavior matches the intended operational behavior or exposes reward-hacking failure modes.
